# Dunnhumby M2 10-seed test-only final run

교수님 피드백에 따라 기존 train과 validation을 합쳐 100 epoch로 고정 학습하고, test는 각 seed·모형의 최종 체크포인트에서 한 번만 평가합니다.

- seeds: 42~51
- models: M1@64, M2, M1@96, shuffled-user
- validation 선택·조기종료 없음, holdout 미사용
- 각 조합은 epoch별 자동 저장되며 재연결 후 다시 실행하면 완료 조합은 재사용하고 미완료 조합만 이어서 실행합니다.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

REVIEWED_SHA = '60ea811a1cc8b191f5a2f54d502e171846d97f3f'
%cd /content
!rm -rf /content/clv-m2-lightgcn-runner
!git clone -q https://github.com/jung-un/clv-m2-lightgcn-runner.git /content/clv-m2-lightgcn-runner
%cd /content/clv-m2-lightgcn-runner
!git checkout -q $REVIEWED_SHA
import subprocess
assert subprocess.check_output(['git', 'rev-parse', 'HEAD'], text=True).strip() == REVIEWED_SHA


In [ ]:
import json
from pathlib import Path
from IPython.display import display
import pandas as pd

from lightgcn_clv_axis_specific_test10 import (
    configure_test10_run, preflight_summary, run_test10,
)

cfg = configure_test10_run(
    out_dir='/content/drive/MyDrive/논문/data/results_v3_dunnhumby_m2_axis_specific_test10_v1',
)
print(json.dumps(preflight_summary(cfg), ensure_ascii=False, indent=2))


## 저장된 진행상태 확인

이 셀은 학습을 시작하지 않습니다. 재연결했을 때 어떤 seed·모형이 완료됐고 어느 epoch까지 저장됐는지 확인할 때 실행합니다.

In [ ]:
progress_files = sorted(Path(cfg.out_dir).glob('progress/*/progress.json'))
if not progress_files:
    print('아직 저장된 progress.json이 없습니다.')
else:
    for progress_path in progress_files:
        print(progress_path)
        print(json.dumps(json.loads(progress_path.read_text()), ensure_ascii=False, indent=2))


## 10시드 일괄 실행

아래 셀 하나가 10개 seed와 네 모형을 모두 순차 실행합니다. 인터넷이 끊기거나 런타임이 재시작되면 위 셀부터 다시 실행한 뒤 이 셀을 다시 실행하세요.

In [ ]:
result_df = run_test10(cfg)


In [ ]:
print('seed별 절대지표 (40행):')
display(result_df)
print('10시드 절대지표 평균·표준편차·95% 구간:')
display(result_df.attrs['absolute_summary'])
print('동일 seed M1@64 대비 개별 차이:')
display(result_df.attrs['paired_seed'])
print('동일 seed M1@64 대비 10시드 평균 차이:')
display(result_df.attrs['paired_summary'])
print('결과 파일:')
print(json.dumps(result_df.attrs['result_paths'], ensure_ascii=False, indent=2))
